In [1]:
!pip install -q torch torchvision torchaudio transformers accelerate sentence-transformers faiss-cpu

In [2]:
import re
import faiss
import pickle
import torch
import numpy as np

from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

c:\Work\Projects\AI\AAI-540-Machine-Learning-Operations\rag_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = "cuda"

embedder = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device=device
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3460.00it/s]


In [4]:
for file in Path("dataset/sb/canto1").rglob("chapter*.txt"):
    print(file)
    print(file.parts)
    break

dataset\sb\canto1\chapter1.txt
('dataset', 'sb', 'canto1', 'chapter1.txt')


In [5]:
def load_docs_bg(base_path):
    docs = []

    for file in Path(base_path).rglob("chapter*.txt"):
        text = file.read_text(encoding="utf-8", errors="ignore").strip()

        parts = str(file).split("\\")

        try:
            scripture = parts[1]
            canto = [p for p in parts if "canto" in p][0]
            chapter = file.stem.replace("chapter", "")
        except:
            scripture = "unknown"
            canto = "unknown"
            chapter = file.stem

        docs.append({
            "text": text,
            "source": scripture,
            "canto": canto,
            "chapter": chapter,
            "path": str(file)
        })

    return docs

In [6]:
from pathlib import Path

def load_docs_sb(base_path):
    docs = []

    for file in Path(base_path).rglob("chapter*.txt"):

        text = file.read_text(
            encoding="utf-8",
            errors="ignore"
        ).strip()

        parts = file.parts

        try:
            scripture = parts[1]   # sb
            canto = parts[2]       # canto1
            chapter = file.stem.replace("chapter", "")

        except Exception:
            scripture = "unknown"
            canto = "unknown"
            chapter = file.stem

        docs.append({
            "text": text,
            "source": scripture,
            "canto": canto,
            "chapter": chapter,
            "path": str(file)
        })

    return docs

In [7]:
bg_docs = load_docs_bg("dataset/bg")
sb_docs = load_docs_sb("dataset/sb/")

print("BG chapters:", len(bg_docs))
print("SB chapters:", len(sb_docs))

BG chapters: 18
SB chapters: 335


In [8]:
def clean_scripture(text):
    text = text.replace("PURPORT", ". ")
    text = text.replace("Synonyms", ". ")
    text = text.replace("TRANSLATION", ". ")
    return text

In [9]:
def chunk_text(text, chunk_size=800, overlap=100):

    sentences = text.split(". ")

    chunks = []
    current = ""

    for s in sentences:
        if len(current) + len(s) < chunk_size:
            current += s + ". "
        else:
            chunks.append(current.strip())
            current = s + ". "

    if current:
        chunks.append(current.strip())

    return chunks

In [10]:
ENTITY_LIST = [
    "Prahlada",
    "Dhruva",
    "Gajendra",
    "Ambarisha",
    "Hiranyakashipu",
    "Narasimha",
    "Narada",
    "Kapila",
    "Bharata"
]

In [11]:
def extract_entities(text):

    found = []

    for e in ENTITY_LIST:
        if e.lower() in text.lower():
            found.append(e)

    return found

In [12]:
def final_clean(text):
    return text.replace("\n", " ").strip()

In [13]:
import re

def expand_verse_ids(verse_id):
    """
    Converts:
    '16' → ['16']
    '16-18' → ['16','17','18']
    """
    if "-" in verse_id:
        start, end = verse_id.split("-")
        return [str(i) for i in range(int(start), int(end) + 1)]
    return [verse_id]

In [14]:
bg_verses = []

for file in Path("dataset/bg").rglob("chapter*.txt"):

    text = file.read_text(encoding="utf-8", errors="ignore")
    text = text.replace("\r", "\n")

    chapter_match = re.search(r"chapter(\d+)", str(file).lower())
    chapter = chapter_match.group(1) if chapter_match else "unknown"

    blocks = re.split(r"(TEXTS?\s+\d+(?:-\d+)?)", text, flags=re.IGNORECASE)

    current_id = None
    current_text = []

    for part in blocks:
        part = part.strip()

        # detect TEXT / TEXTS / TEXT 16-18
        match = re.match(r"TEXTS?\s+(\d+(?:-\d+)?)", part, re.IGNORECASE)

        if match:
            # flush previous verse
            if current_id and current_text:
                text_joined = " ".join(current_text).strip()

                for vid in expand_verse_ids(current_id):
                    bg_verses.append({
                        "chapter": chapter,
                        "verse_id": vid,
                        "text": text_joined,
                        "entities": extract_entities(text_joined)
                    })

            current_id = match.group(1)
            current_text = []

        else:
            current_text.append(part)

    # flush last block
    if current_id and current_text:
        text_joined = " ".join(current_text).strip()

        for vid in expand_verse_ids(current_id):
            bg_verses.append({
                "chapter": chapter,
                "verse_id": vid,
                "text": text_joined,
                "entities": extract_entities(text_joined)
            })
print("BG verses:", len(bg_verses))

BG verses: 700


In [15]:
chunked_sb_docs = []

for doc in sb_docs:

    text = clean_scripture(doc["text"])

    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):

        chunked_sb_docs.append({
            **doc,
            "chunk_id": i,
            "text": chunk,
            "entities": extract_entities(chunk)
        })

print("SB chunks:", len(chunked_sb_docs))

SB chunks: 24337


In [16]:
# --- ALIGNMENT SAFE BG DATA PREPARATION ---
bg_texts = []
bg_index_map = []

for d in bg_verses:
    text = final_clean(d["text"])

    if isinstance(text, str) and len(text.strip()) > 30:
        bg_texts.append(text)
        bg_index_map.append(d)

print("Total BG texts:", len(bg_texts))

# --- batch encoding (faster + stable) ---
bg_embeddings_list = []

batch_size = 128

def batch_iter(data, batch_size):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

for i, batch in enumerate(batch_iter(bg_texts, batch_size)):

    emb = embedder.encode(
        batch,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    bg_embeddings_list.append(emb)

    print(f"BG batch {i+1} done")

# --- merge ---
bg_embeddings = np.vstack(bg_embeddings_list)

print("BG embedding shape:", bg_embeddings.shape)

# --- FAISS index ---
bg_index = faiss.IndexFlatIP(bg_embeddings.shape[1])
bg_index.add(bg_embeddings)

print("✅ BG TEXTS COUNT:", len(bg_texts))
print("SAMPLE TEXT:", bg_texts[0][:200])

print("✅ FAISS SIZE:", bg_index.ntotal)

print("BG FAISS index built successfully")

Total BG texts: 700
BG batch 1 done
BG batch 2 done
BG batch 3 done
BG batch 4 done
BG batch 5 done
BG batch 6 done
BG embedding shape: (700, 768)
✅ BG TEXTS COUNT: 700
SAMPLE TEXT: Translation Dhrtarashtra said: O Sanjaya, after my sons and the sons of Pandu assembled in the place of pilgrimage at Kurukshetra, desiring to fight, what did they do?  Purport Bhagavad-gita is the wi
✅ FAISS SIZE: 700
BG FAISS index built successfully


In [17]:

# --- OPTIMIZED SB EMBEDDING (FAST GPU BATCH) ---

sb_texts = []
sb_index_map = []

for d in chunked_sb_docs:

    text = final_clean(d["text"])

    if isinstance(text, str) and len(text.strip()) > 80:
        sb_texts.append(text)
        sb_index_map.append(d)

print("Total SB texts:", len(sb_texts))

# Direct batch encoding (NO MANUAL LOOP)
sb_embeddings = embedder.encode(
    sb_texts,
    batch_size=256,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("SB embedding shape:", sb_embeddings.shape)

# FAISS index
sb_index = faiss.IndexFlatIP(sb_embeddings.shape[1])
sb_index.add(sb_embeddings)

print("SB FAISS index built successfully")


Total SB texts: 24329
SB embedding shape: (24329, 768)
SB FAISS index built successfully


In [18]:
faiss.write_index(bg_index, "bg.faiss")
faiss.write_index(sb_index, "sb.faiss")

with open("bg_docs.pkl", "wb") as f:
    pickle.dump(bg_verses, f)

with open("sb_docs.pkl", "wb") as f:
    pickle.dump(chunked_sb_docs, f)

In [19]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:03<00:00, 105.16it/s]


In [20]:
def search_bg(question, k):
    q_emb = embedder.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    # retrieve more candidates for reranking
    scores, ids = bg_index.search(q_emb, k * 10)

    results = []

    #q_lower = question.lower()

    for score, idx in zip(scores[0], ids[0]):

        doc = bg_index_map[idx]

        #text = doc["text"]
        #text_lower = text.lower()

        final_score = float(score)

        # -----------------------------
        # LIGHT DOCTRINE-AWARE BOOST (NO HARD CODING ANSWERS)
        # -----------------------------


        # general Bhagavad Gita instruction boost (Bhagavad Gita conclusion chapters)
        #if doc.get("chapter") in ["18"]:
         #   final_score += 0.3

        results.append({
            "score": final_score,
            "chapter": doc.get("chapter", "unknown"),
            "verse_id": doc.get("verse_id", "unknown"),
            "text": doc["text"]
        })

    # sort by adjusted score
    results.sort(key=lambda x: x["score"], reverse=True)
    
    #print("results:", results)

    #print("Top BG scores:", [round(r["score"], 3) for r in results[:k]])
    #print("Verse:", [r["chapter"] + "." + r["verse_id"] for r in results[:k]])

    return results[:k]

In [21]:
def search_sb(question, k):

    q_emb = embedder.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    scores, ids = sb_index.search(q_emb, k * 3)

    results = []

    #print("\n================ SB RETRIEVAL DEBUG ================\n")

    for rank, (idx, score) in enumerate(zip(ids[0], scores[0])):

        doc = sb_index_map[idx]

        item = {
            "rank": rank + 1,
            "score": float(score),
            "chapter": doc.get("chapter"),
            "canto": doc.get("canto"),
            "text": doc["text"],
            "entities": doc.get("entities", []),
            "path": doc.get("path")
        }

        results.append(item)

        # 🔍 DEBUG VIEW
        #print(f"Rank {rank+1}")
        #print(f"Score: {float(score):.4f}")
        #print(f"Canto: {item['canto']}")
        #print(f"Chapter: {item['chapter']}")
        #print("Text preview:")
        #print(item["text"][:300])
        #print("-" * 60)

    #print("\n================ FINAL RANKING ================\n")

    #for i, r in enumerate(results[:k], 1):
        #print(f"{i}. {r['canto']} | {r['chapter']} | score={r['score']:.4f}")

    return results[:k]

In [22]:
def explain_retrieval(question, results, top_k=3):

    print("========== 📖 SRIMAD BHAGAVATAM RESULTS =============")

    print(f"Query: {question}\n")

    for r in results[:top_k]:

        print(f"Rank {r['rank']}")
        print(f"Similarity Score: {r['score']:.3f}")
        print(f"Source: {r['canto']} | Chapter {r['chapter']}")

        print("Why this was retrieved:")
        
        # simple explanation heuristic
        matched_words = [
            w for w in question.lower().split()
            if w in r["text"].lower()
        ]

        if matched_words:
            print(f"- Keyword overlap: {matched_words}")
        else:
            print("- Semantic similarity (no direct keyword match)")

        #print("\nExcerpt:")
        #print(r["text"][:300])

        print("--------------------------------------------------")

In [23]:
def classify_query(question):
    q = question.lower()

    if any(w in q for w in ["surrender", "take refuge", "submit", "whom shall", "whom should"]):
        return "philosophy"

    if any(w in q for w in ["story", "devotee", "example", "pastime", "incident"]):
        return "narrative"

    return "general"

In [24]:
def clean(text, max_chars=900):
    return text.replace("\n", " ")[:max_chars]

In [25]:
def format_answer(question, bg_results, sb_results):

    bg = bg_results[0]  # best match
    print("All results:", bg_results)

    bg_block = f"""
📜 BHAGAVAD GITA (PRIMARY)

Chapter: {bg.get('chapter', '?')}
Verse: {bg.get('verse_id', '?')}

Text:
{bg['text'][:300]}...

"""

    sb_block = "\n\n".join([
        f"""
📖 SRIMAD BHAGAVATAM

Canto: {s.get('canto', '?')}
Chapter: {s.get('chapter', '?')}

Text:
{s['text'][:250]}...
"""
        for s in sb_results
    ])

    final = f"""
══════════════════════════════════════
QUESTION:
{question}
══════════════════════════════════════

{bg_block}

{sb_block}

══════════════════════════════════════
🎯 FINAL CONCLUSION

Based on Bhagavad Gita and Srimad Bhagavatam:
Lord Krishna is the ultimate shelter (BG 18.66).
══════════════════════════════════════
"""

    return final

In [26]:
def summarize_chunk(text):
    prompt = f"""
Summarize the spiritual teaching of this passage in 1-2 lines.
Focus only on the core instruction or philosophy.

TEXT:
{text}

SUMMARY:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")

    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=50, do_sample=False)

    return tokenizer.decode(out[0], skip_special_tokens=True)

In [27]:

def ask(question):

    mode = classify_query(question)

    # -------------------------
    # RETRIEVAL ROUTING
    # -------------------------

        
    bg_results = search_bg(question, k=2)
    sb_results = search_sb(question, k=2)

    explain_retrieval(question, sb_results)
    
    bg_context = "\n\n".join(clean(r["text"]) for r in bg_results)
    sb_context = "\n\n".join(clean(r["text"]) for r in sb_results)

    # -------------------------
    # LLM PROMPT (NO FORMATTING RESPONSIBILITY)
    # -------------------------
    prompt = f"""
You are a spiritual knowledge assistant.

Use ONLY the provided context.

Bhagavad Gita is the primary authority.
Srimad Bhagavatam is supporting explanation.

Do NOT format the answer.
Do NOT add headings or structure.

Just explain the meaning clearly and concisely.

QUESTION:
{question}

BG CONTEXT:
{bg_context}

SB CONTEXT:
{sb_context}

ANSWER:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    with torch.no_grad():
        out = llm.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False,
            temperature=0.2
        )

    llm_output = tokenizer.decode(out[0], skip_special_tokens=True)

    # -------------------------
    # FINAL STRUCTURED FORMATTING (PYTHON CONTROLLED)
    # -------------------------
    bg_block = ""
    if bg_results:
        bg = bg_results[0]
        bg_block = f"""
Chapter: {bg.get('chapter', '?')}
Verse: {bg.get('verse_id', '?')}

Text:
{clean(bg['text'])[:400]}...
"""

    sb_block = "\n\n".join([
        f"""
📖 SRIMAD BHAGAVATAM

Canto: {s.get('canto', '?')}
Chapter: {s.get('chapter', '?')}

Text:
{clean(s['text'])[:250]}...
"""
        for s in sb_results
    ])

    final_output = f"""
========== 📖 BHAGAVADGITA RESULTS =============
{bg_block}
🎯 EXPLANATION (LLM INSIGHT)

{llm_output}

══════════════════════════════════════
"""

    return final_output

In [29]:
import time

#question = "What are the qualities of a devotee?"
question = "What krishna instructs shall we ultimately surrender unto?"
 
start = time.time()

print(ask(question))

print("\nTime taken:", time.time() - start)

========== 📖 SRIMAD BHAGAVATAM RESULTS =============
Query: What krishna instructs shall we ultimately surrender unto?

Rank 1
Similarity Score: 0.751
Source: canto11 | Chapter 11
Why this was retrieved:
- Keyword overlap: ['krishna', 'ultimately', 'surrender']
--------------------------------------------------
Rank 2
Similarity Score: 0.750
Source: canto2 | Chapter 6
Why this was retrieved:
- Keyword overlap: ['krishna', 'surrender']
--------------------------------------------------

========== 📖 BHAGAVADGITA RESULTS =============

Chapter: 18
Verse: 66

Text:
Translation Abandon all varieties of religion and just surrender unto Me. I shall deliver you from all sinful reactions. Do not fear.  Purport The Lord has described various kinds of knowledge and processes of religion – knowledge of the Supreme Brahman, knowledge of the Supersoul, knowledge of the different types of orders and statuses of social life, knowledge of the renounced order of life, kno...

🎯 EXPLANATION (LLM INSIGHT